In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# raiz="drive/MyDrive/proyecto_mineria/"
raiz=""

# Cargar datos y generar los conjuntos de entrenamiento y test

In [2]:
import pandas as pd
nombre_csv_logs="presentacion1_resultados.csv"
# file_path = 'data_set_limpio.pkl'
file_path = f'{raiz}datasets_pkl/data_set_limpio_sin_not_for_sale.pkl'

df = pd.read_pickle(file_path)

# Display the loaded DataFrame
print("shape: ",df.shape)
print(df.columns)
df.sample(n=5)


shape:  (186795, 70)
Index(['Nat', 'Division', 'Club', 'Based', 'Preferred Foot', 'Right Foot',
       'Left Foot', 'Position', 'Height', 'Weight', 'Age', 'Wage', 'AT Apps',
       'AT Gls', 'Team', 'Caps', 'Yth Apps', 'Style', 'Rc Injury', 'Best Role',
       'Best Duty', 'Best Pos', 'Acc', 'Aer', 'Agg', 'Agi', 'Ant', 'Bal',
       'Bra', 'Cmd', 'Com', 'Cmp', 'Cnt', 'Cor', 'Cro', 'Dec', 'Det', 'Dri',
       'Ecc', 'Fin', 'Fir', 'Fla', 'Fre', 'Han', 'Hea', 'Jum', 'Kic', 'Ldr',
       'Lon', 'L Th', 'Mar', 'Nat .1', 'OtB', '1v1', 'Pac', 'Pas', 'Pen',
       'Pos', 'Pun', 'Ref', 'TRO', 'Sta', 'Str', 'Tck', 'Tea', 'Tec', 'Thr',
       'Vis', 'Wor', 'transfer_value_estimado'],
      dtype='object')


,Nat,Division,Club,Based,Preferred Foot,Right Foot,Left Foot,Position,Height,Weight,...,TRO,Sta,Str,Tck,Tea,Tec,Thr,Vis,Wor,transfer_value_estimado
134392,CHN,Chinese Super League,Guangzhou,China (Super League),Right Only,Very Strong,Weak,"M/AM (RL), ST (C)",176,70,...,3,9,7,6,12,6,3,8,13,260000
7035,COL,Colombian Second Division,Boca Juniors de Cali,Colombia (Second Division),Right,Very Strong,Weak,D (C),186,76,...,3,7,8,10,7,2,3,1,8,385000
185268,FRA,Ligue 1 Uber Eats,LOSC,France (Ligue 1 Uber Eats),Right Only,Very Strong,Weak,"AM (C), ST (C)",171,59,...,2,12,5,1,6,13,2,11,7,99000
52717,BIH,Swiss First League Group 3,Lugano,Switzerland (First League Group 3),Right,Very Strong,Weak,D (C),190,80,...,1,8,6,13,5,5,2,1,6,11000
47430,FRA,Jupiler Pro League,Oostende,Belgium (Jupiler Pro League),Left Only,Weak,Very Strong,D/WB/M/AM (L),169,65,...,4,11,9,10,11,12,2,12,16,1762500


# Generar conjuntos de entrenamiento y test

In [3]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def log_results(model, method_name, X_train, y_train, X_test, y_test, filepath=f"{raiz}results_regressionCV.csv"):
    # parameters
    params_dict = model.get_params()
    
    # Convert non-serializable objects
    for k, v in params_dict.items():
        try:
            json.dumps(v)
        except TypeError:
            params_dict[k] = str(v)

    params = json.dumps(params_dict)
    # Predictions
    y_pred = model.predict(X_test)
    # Metrics
    r2_train = model.score(X_train, y_train)
    r2_test = model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # Row to store
    row = {
        "method": method_name,
        "hyperparameters": json.dumps(params),
        "r2_train": r2_train,
        "r2_test": r2_test,
        "mae": mae,
        "rmse": rmse
    }
    # Append or create CSV
    try:
        df = pd.read_csv(filepath)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    except FileNotFoundError:
        df = pd.DataFrame([row])
    print("si pase")
    df.to_csv(filepath, index=False)

# Generar conjuntos de entrenamiento y test

In [4]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [5]:
import clustering as cl

kmeans_model_club = cl.fit_kmeans(train_df, columna="Club")
kmeans_model_nat = cl.fit_kmeans(train_df, columna="Nat")
kmeans_model_division = cl.fit_kmeans(train_df, columna="Division")


train_df = cl.apply_kmeans(train_df, kmeans_model_club)
train_df = cl.apply_kmeans(train_df, kmeans_model_nat)
train_df = cl.apply_kmeans(train_df, kmeans_model_division)

test_df = cl.apply_kmeans(test_df, kmeans_model_club)
test_df = cl.apply_kmeans(test_df, kmeans_model_nat)
test_df = cl.apply_kmeans(test_df, kmeans_model_division)



In [6]:
target="transfer_value_estimado"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

## One Hot para las features categóricas

In [7]:
# import numpy as np
# # categorical_cols=["Nat_cluster","Division_cluster","Club_cluster","Preferred Foot","Right Foot","Left Foot","Best Pos","Best Duty","Style","Best Role","Rc Injury"]
# categorical_cols=["Nat_cluster","Division_cluster","Best Duty","Style"]
# X_train = pd.get_dummies(X_train, columns=categorical_cols)
# X_test  = pd.get_dummies(X_test, columns=categorical_cols)

# X_train, X_test = X_train.align(X_test, join='left', axis=1,fill_value=0)

In [8]:
# X_train = X_train.select_dtypes(include=[np.number,np.bool_])
# X_test = X_test.select_dtypes(include=[np.number,np.bool_])

In [9]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()

# X_train = pd.DataFrame(
#     scaler.fit_transform(X_train),
#     columns=X_train.columns,
#     index=X_train.index
# )

# X_test = pd.DataFrame(
#     scaler.transform(X_test),
#     columns=X_test.columns,
#     index=X_test.index
# )

# Entrenamiento

In [10]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Identify categorical columns
cat_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print(f"Categorical columns: {cat_cols}")

# 2. Copy datasets
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# 3. Encode all categorical columns
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_train_encoded[cat_cols] = encoder.fit_transform(
    X_train[cat_cols]
)

X_test_encoded[cat_cols] = encoder.transform(
    X_test[cat_cols]
)

# 4. Train model
model = HistGradientBoostingRegressor()

# 

Categorical columns: ['Nat', 'Division', 'Club', 'Based', 'Preferred Foot', 'Right Foot', 'Left Foot', 'Position', 'Team', 'Style', 'Rc Injury', 'Best Role', 'Best Duty', 'Best Pos']


# Entrenamiento y Evaluación

In [11]:
model.fit(X_train_encoded, y_train)
# model.fit(X_train_encoded, y_train)

# # 5. Predict
preds = model.predict(X_test_encoded)
log_results(model,"HistGradientBoostingRegressor",X_train_encoded,y_train,X_test_encoded,y_test,filepath=nombre_csv_logs)

si pase


In [12]:
preds = model.predict(X_test_encoded)
print("saco ",model.score(X_test_encoded,y_test))
preds

saco  0.6774620271021263


array([283297.02203993, -34024.5438806 ,   2792.30636065, ...,
        49902.13785053,   4291.32260243,  21887.4530459 ], shape=(37359,))

In [13]:
from sklearn.inspection import permutation_importance
import pandas as pd

result = permutation_importance(
    model,          # trained HistGradientBoostingRegressor
    X_test_encoded,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance_df = pd.DataFrame({
    "feature": X_test_encoded.columns,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
})

importance_df = importance_df.sort_values(
    "importance_mean",
    ascending=False
)

print(importance_df)

   feature  importance_mean  importance_std
11    Wage    372972.481975     3039.354405
10     Age    233108.147624     6887.912670
3    Based     25320.359140     2623.899489
2     Club      6331.660209     1565.037213
22     Acc      6237.803682     1791.197465
..     ...              ...             ...
46     Kic      -742.296168      144.157354
43     Han      -836.785394      333.703013
51  Nat .1     -1592.767973      457.168509
34     Cro     -2255.283882      517.763845
32     Cnt     -2636.733882      573.625979

[72 rows x 3 columns]


In [ ]:
print(importance_df)

# Grid search

In [14]:
# from sklearn.model_selection import GridSearchCV
# param_grid = {
#     'regressor__max_iter': [100, 200],
#     'regressor__learning_rate': [0.01, 0.1],
#     'regressor__max_leaf_nodes': [31, 63],
#     # You can even tune the preprocessor!
#     'prep__high_card__smooth': ['auto', 1.0] 
# }
# gbr_cv = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1, scoring='neg_mean_squared_error')
# gbr_cv.fit(X_train, y_train)

## Evaluación

In [15]:
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# y_pred = gbr_cv.predict(X_test)
# mae=mean_absolute_error(y_test, y_pred)
# mse=mean_squared_error(y_test, y_pred)
# r2s=r2_score(y_test, y_pred)

# print("Best parameters ",gbr_cv.best_params_)
# print("mean_absolute_error: ",mae)
# print("mean_squared_error: ",mse)
# print("r2_score: ",r2s)